# Tutorial Interactivo de DeepEval
Este notebook te guiará a través de las pruebas unitarias para LLMs usando DeepEval y modelos de Google Generative AI.

In [ ]:
!pip install -q deepeval google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.1/965.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.8 MB/s eta 0:00:00


## 1. Configuración del Modelo de Evaluación
Configuraremos a Gemini como nuestro evaluador. DeepEval permite usar modelos personalizados definiendo una clase que herede de `DeepEvalBaseLLM`.

In [ ]:
import google.generativeai as genai
from deepeval.models.base_model import DeepEvalBaseLLM
from google.colab import userdata

# Configurar la API Key desde los secretos de Colab
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

class GeminiModel(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-3.1-flash-lite"):
        self.model = genai.GenerativeModel(model_name)

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = chat_model.generate_content(prompt)
        return res.text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "gemini-3.1-flash-lite"

evaluator_model = GeminiModel()
print("Modelo evaluador (Gemini Lite) configurado correctamente.")

Modelo evaluador (Gemini Lite) configurado correctamente.


## 2. Nuestra Primera Métrica: Faithfulness
La métrica de Fidelidad mide si el output del LLM se deriva puramente del contexto proporcionado. Es fundamental para evitar alucinaciones en sistemas RAG.

In [ ]:
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# 1. Definimos el caso de prueba
input_text = "¿Cuál es la política de devoluciones?"
context = ["Nuestra política permite devoluciones en un plazo de 30 días, siempre que el producto esté en su empaque original."]
actual_output = "Puedes devolver tus productos en un plazo de 30 días si conservas el empaque original."

test_case = LLMTestCase(
    input=input_text,
    actual_output=actual_output,
    retrieval_context=context
)

# 2. Configuramos la métrica usando nuestro evaluador Gemini
metric = FaithfulnessMetric(
    threshold=0.7,
    model=evaluator_model,
    include_reason=True
)

# 3. Ejecutamos la evaluación
metric.measure(test_case)

print(f"Puntaje de Fidelidad: {metric.score}")
print(f"Razón del puntaje: {metric.reason}")

Output()

Puntaje de Fidelidad: 1.0
Razón del puntaje: The score is 1.00 because the actual output is perfectly consistent with the retrieval context—fantastic job maintaining such high accuracy!


## 3. Métrica: Answer Relevancy
Esta métrica evalúa si la respuesta del modelo es concisa y aborda directamente la intención del usuario.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric

# Reutilizamos el test_case anterior
relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=evaluator_model,
    include_reason=True
)

relevancy_metric.measure(test_case)

print(f"Puntaje de Relevancia: {relevancy_metric.score}")
print(f"Razón: {relevancy_metric.reason}")

Output()

Puntaje de Relevancia: 1.0
Razón: The score is 1.00 because the response provided a perfectly direct and accurate explanation of the return policy without including any unnecessary or off-topic information. Great job!


## 4. Evaluaciones en Lote (Batch Testing)
En un entorno real, no probarás una sola respuesta. DeepEval permite definir un conjunto de datos y evaluarlos todos de una vez.

In [ ]:
from deepeval import evaluate

# Definimos varios casos de prueba
case_1 = LLMTestCase(
    input="¿Tienen envíos internacionales?",
    actual_output="Sí, enviamos a todo el mundo.",
    retrieval_context=["Realizamos envíos nacionales e internacionales (excepto a zonas de conflicto)."]
)

case_2 = LLMTestCase(
    input="¿Cuál es el precio del iPhone 15?",
    actual_output="El iPhone 15 cuesta 999 dólares.",
    retrieval_context=["Los precios de los smartphones varían según la región, consulta el catálogo local."]
)

# Ejecutamos una evaluación masiva
evaluate([case_1, case_2], metrics=[relevancy_metric, metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-3.1-flash-lite, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-3.1-flash-lite, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Answer Relevancy                      │ 1.00                           │ 100.00%               │ 2             │
│  Faithfulness                          │ 1.00                           │ 100.00%               │ 2             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=699107;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 23.88s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason="The score is 1.00 because you provided a perfectly clear and relevant answer that directly addresses the customer's inquiry.", strict_mode=False, evaluation_model='gemini-3.1-flash-lite', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "Sí.",\n    "Enviamos a todo el mundo."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the actual output is perfectly consistent with the retrieval context—fantastic job maintaining high accuracy!', strict_mode=False, evaluation_model='gemini-3.1-flash-lite', error=None, evaluation_cost=None, verbose_logs='Truths (limit=None):\n[\n    "La empresa real

## 5. Métrica Personalizada: G-Eval
G-Eval es una métrica basada en LLM que permite definir criterios subjetivos o específicos de negocio mediante lenguaje natural.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

# Definimos una métrica de 'Tono Profesional'
tone_metric = GEval(
    name="Tono Profesional",
    criteria="Determina si el output es profesional, educado y no utiliza jerga informal.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=evaluator_model,
    threshold=0.7
)

custom_test_case = LLMTestCase(
    input="¿Cómo va mi pedido?",
    actual_output="¡Qué onda! Tu pedido ya va en camino, tranqui."
)

# Intentamos medir (recuerda esperar si el error de cuota persiste)
try:
    tone_metric.measure(custom_test_case)
    print(f"Puntaje de Tono: {tone_metric.score}")
    print(f"Razón: {tone_metric.reason}")
except Exception as e:
    print(f"Error: {e}. Por favor espera un momento antes de reintentar.")

Output()

Puntaje de Tono: 0.0
Razón: El texto utiliza un lenguaje extremadamente informal y coloquial (expresiones como '¡Qué onda!' y 'tranqui'), lo cual contraviene totalmente los estándares de comunicación corporativa y profesional requeridos por las instrucciones de evaluación.


## Conclusión
¡Felicidades! Has aprendido a:
1. Configurar un evaluador personalizado (Gemini).
2. Medir la Fidelidad (Faithfulness) para evitar alucinaciones.
3. Medir la Relevancia de las respuestas.
4. Ejecutar pruebas en lote (Batch).
5. Crear métricas personalizadas con G-Eval.